# 🌬️ Tempêtes européennes — StormEurope (CLIMADA)

Exploration du module `StormEurope` de CLIMADA pour modéliser les tempêtes hivernales en Europe du Nord-Ouest.

**Aléa** : Tempête européenne (WS)  
**Zone** : Europe du Nord-Ouest (France, Allemagne, UK)  
**Données** : CDS WISC footprints / synthétique  
**Métriques** : SSI (Storm Severity Index), EAI, courbes de fréquence

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from climada.hazard import Centroids
from climada.entity import Exposures, ImpactFuncSet, ImpactFunc
from climada.engine import Impact

# Petals
try:
    from climada.hazard import StormEurope
    from climada.entity.impact_funcs.storm_europe import ImpfStormEurope
    PETALS_OK = True
except ImportError:
    PETALS_OK = False
    print("⚠️ climada_petals non disponible — on utilisera des données synthétiques")

print(f"CLIMADA petals disponible : {PETALS_OK}")

## 1. Aléa — Tempêtes européennes

Les tempêtes extratropicales (windstorms) sont le 2ᵉ péril le plus coûteux en Europe après les inondations. CLIMADA utilise les **footprints WISC** (Windstorm Information Service from Copernicus) ou des footprints synthétiques.

### Données WISC (CDS)
Le dataset contient ~150 tempêtes historiques (1940-2014) sous forme de grilles de rafales max (m/s).

In [ ]:
# --- Téléchargement WISC depuis CDS (nécessite compte CDS) ---
# from climada.hazard import StormEurope
# storm = StormEurope.from_footprints(
#     path='/path/to/wisc/footprints/',
#     centroids=centroids,
#     intensity_thres=15.0,  # m/s seuil minimum
#     description='WISC historical storms'
# )

# --- Données synthétiques pour démonstration ---
from climada.hazard import Hazard
from scipy.sparse import csr_matrix

np.random.seed(42)

# Grille Europe NW : 45-60°N, -10 à 15°E
lon = np.arange(-10, 15.5, 0.5)
lat = np.arange(45, 60.5, 0.5)
lon_grid, lat_grid = np.meshgrid(lon, lat)
lon_flat = lon_grid.flatten()
lat_flat = lat_grid.flatten()
n_centroids = len(lon_flat)

centroids = Centroids.from_lat_lon(lat_flat, lon_flat)
print(f"Centroids : {centroids.size} points sur l'Europe NW")

# Générer 30 tempêtes synthétiques
n_storms = 30
frequency = np.ones(n_storms) / 30  # ~1/an sur 30 ans

intensity_list = []
for i in range(n_storms):
    # Centre aléatoire de la tempête
    center_lon = np.random.uniform(-5, 10)
    center_lat = np.random.uniform(48, 57)
    
    # Distance au centre
    dist = np.sqrt((lon_flat - center_lon)**2 + (lat_flat - center_lat)**2)
    
    # Profil de vent : max au centre, décroissance gaussienne
    max_wind = np.random.uniform(25, 55)  # m/s
    radius = np.random.uniform(3, 8)  # degrés
    wind = max_wind * np.exp(-0.5 * (dist / radius)**2)
    
    # Seuil : < 15 m/s = pas de dommage
    wind[wind < 15] = 0
    intensity_list.append(csr_matrix(wind))

from scipy.sparse import vstack
intensity_matrix = vstack(intensity_list)

storm_haz = Hazard(
    haz_type='WS',
    centroids=centroids,
    intensity=intensity_matrix,
    frequency=frequency,
    event_id=np.arange(1, n_storms + 1),
    event_name=[f'Storm_{i+1:03d}' for i in range(n_storms)],
    date=pd.date_range('1985-01-01', periods=n_storms, freq='365D').to_numpy().astype('datetime64[ns]').astype(int) // 10**9,
    units='m/s'
)

print(f"Hazard : {storm_haz.size} événements, intensité max = {storm_haz.intensity.max():.1f} m/s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Tempête la plus intense
max_idx = np.array(storm_haz.intensity.max(axis=1).todense()).flatten().argmax()
max_intensity = storm_haz.intensity[max_idx].toarray().flatten()

sc1 = axes[0].scatter(lon_flat, lat_flat, c=max_intensity, s=2, cmap='YlOrRd', vmin=0)
axes[0].set_title(f'Tempête la plus intense : {storm_haz.event_name[max_idx]}')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
plt.colorbar(sc1, ax=axes[0], label='Vent max (m/s)')

# Intensité maximale sur toutes les tempêtes
max_all = np.array(storm_haz.intensity.max(axis=0).todense()).flatten()
sc2 = axes[1].scatter(lon_flat, lat_flat, c=max_all, s=2, cmap='YlOrRd', vmin=0)
axes[1].set_title('Vent max toutes tempêtes confondues')
axes[1].set_xlabel('Longitude')
plt.colorbar(sc2, ax=axes[1], label='Vent max (m/s)')

plt.tight_layout()
plt.show()

## 2. Storm Severity Index (SSI)

Le **SSI** quantifie la sévérité d'une tempête en intégrant la surface touchée et l'intensité au-delà d'un seuil. C'est une métrique standard en réassurance.

$$SSI = \sum_{i} \left(\frac{v_i}{v_{98}}\right)^3 \cdot A_i$$

Où $v_i$ est la vitesse du vent, $v_{98}$ le 98ᵉ percentile local, et $A_i$ l'aire de la cellule.

In [ ]:
# Calcul du SSI simplifié
# Seuil = 98e percentile local (ici on utilise un seuil fixe de 20 m/s)
v_threshold = 20.0  # m/s

ssi_values = []
for i in range(n_storms):
    wind_i = storm_haz.intensity[i].toarray().flatten()
    mask = wind_i > v_threshold
    if mask.any():
        # Cube de l'excès normalisé × nombre de cellules touchées
        excess = (wind_i[mask] / v_threshold) ** 3
        ssi = excess.sum()
    else:
        ssi = 0
    ssi_values.append(ssi)

ssi_df = pd.DataFrame({
    'event': storm_haz.event_name,
    'max_wind_ms': np.array(storm_haz.intensity.max(axis=1).todense()).flatten(),
    'ssi': ssi_values
}).sort_values('ssi', ascending=False)

print("Top 10 tempêtes par SSI :")
print(ssi_df.head(10).to_string(index=False))

# Visualisation
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(ssi_df)), ssi_df['ssi'].values, color='steelblue', alpha=0.8)
ax.set_xlabel('Tempête (triée par SSI)')
ax.set_ylabel('SSI')
ax.set_title('Storm Severity Index — Tempêtes synthétiques Europe NW')
plt.tight_layout()
plt.show()

## 3. Exposition — LitPop Europe NW

`LitPop` combine luminosité nocturne (nightlights) et population pour estimer la valeur des actifs exposés à résolution ~1 km.

In [ ]:
# --- LitPop (nécessite téléchargement) ---
# from climada.entity import LitPop
# exposure = LitPop.from_countries(
#     countries=['FRA', 'DEU', 'GBR'],
#     res_arcsec=150,  # ~5 km pour accélérer
#     fin_mode='gdp'
# )

# --- Exposition synthétique ---
from shapely.geometry import Point

n_assets = 500
np.random.seed(123)

# Distribution pondérée vers les zones urbaines
cities = {
    'Paris': (2.35, 48.86, 0.25),
    'London': (-0.12, 51.51, 0.20),
    'Berlin': (13.40, 52.52, 0.12),
    'Amsterdam': (4.90, 52.37, 0.08),
    'Brussels': (4.35, 50.85, 0.07),
    'Hamburg': (9.99, 53.55, 0.05),
    'Frankfurt': (8.68, 50.11, 0.05),
    'Lyon': (4.83, 45.76, 0.05),
    'Manchester': (-2.24, 53.48, 0.05),
    'Cologne': (6.95, 50.94, 0.04),
    'Dublin': (-6.27, 53.35, 0.04),
}

lons, lats, values = [], [], []
for city, (clon, clat, weight) in cities.items():
    n = int(n_assets * weight)
    lons.extend(np.random.normal(clon, 0.5, n))
    lats.extend(np.random.normal(clat, 0.3, n))
    base_value = np.random.uniform(5e6, 50e6)
    values.extend(np.random.exponential(base_value, n))

# Compléter si nécessaire
remaining = n_assets - len(lons)
if remaining > 0:
    lons.extend(np.random.uniform(-5, 12, remaining))
    lats.extend(np.random.uniform(46, 58, remaining))
    values.extend(np.random.exponential(10e6, remaining))

import geopandas as gpd

gdf = gpd.GeoDataFrame({
    'value': np.array(values[:n_assets]),
    'impf_WS': np.ones(n_assets, dtype=int),
    'geometry': [Point(lo, la) for lo, la in zip(lons[:n_assets], lats[:n_assets])]
}, crs='EPSG:4326')

exposure = Exposures(gdf)
exposure.check()

print(f"Exposition : {len(exposure.gdf)} actifs")
print(f"Valeur totale : {exposure.gdf['value'].sum():,.0f} USD")

# Carte
fig, ax = plt.subplots(figsize=(10, 8))
exposure.gdf.plot(ax=ax, column='value', markersize=5, cmap='YlOrRd', 
                  legend=True, legend_kwds={'label': 'Valeur (USD)', 'shrink': 0.6})
ax.set_title("Exposition synthétique — Europe NW")
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

## 4. Vulnérabilité — Fonctions d'impact tempête

CLIMADA propose deux calibrations pour les tempêtes européennes :
- **Schwierz (2010)** : sigmoïde calibrée sur les pertes suisses
- **Welker (2021)** : calibration mise à jour, recommandée

La variable d'entrée est la vitesse de rafale (m/s).

In [ ]:
# --- Fonctions d'impact CLIMADA (si petals disponible) ---
if PETALS_OK:
    try:
        impf_schwierz = ImpfStormEurope.from_schwierz()
        impf_welker = ImpfStormEurope.from_welker()
        print("Fonctions Schwierz et Welker chargées depuis CLIMADA")
    except Exception as e:
        print(f"Erreur chargement : {e}")
        PETALS_OK = False

# --- Fonctions custom (fallback ou comparaison) ---
# Schwierz-like : sigmoïde
intensity_range = np.arange(0, 80, 0.5)

# Sigmoïde Schwierz (approximation)
v_half = 35  # m/s à 50% de dommage
k = 0.15     # pente
mdr_schwierz = 1 / (1 + np.exp(-k * (intensity_range - v_half)))
mdr_schwierz[intensity_range < 15] = 0  # pas de dommage sous 15 m/s

# Welker : cube normalisé (plus réaliste pour les vents forts)
v_ref = 25  # m/s seuil
mdr_welker = np.clip(((intensity_range - v_ref) / 40) ** 3, 0, 1)
mdr_welker[intensity_range < v_ref] = 0

# Créer les ImpactFunc CLIMADA
impf_schwierz_custom = ImpactFunc(
    id=1,
    haz_type='WS',
    intensity=intensity_range,
    mdd=mdr_schwierz,
    paa=np.ones_like(mdr_schwierz),
    intensity_unit='m/s',
    name='Schwierz (approx.)'
)

impf_welker_custom = ImpactFunc(
    id=2,
    haz_type='WS',
    intensity=intensity_range,
    mdd=mdr_welker,
    paa=np.ones_like(mdr_welker),
    intensity_unit='m/s',
    name='Welker (approx.)'
)

# Visualisation
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(intensity_range, mdr_schwierz, 'b-', linewidth=2, label='Schwierz (sigmoïde)')
ax.plot(intensity_range, mdr_welker, 'r--', linewidth=2, label='Welker (cube)')
ax.axvline(x=25, color='gray', linestyle=':', alpha=0.5, label='Seuil dommage (25 m/s)')
ax.set_xlabel('Vitesse du vent (m/s)')
ax.set_ylabel('MDR (Mean Damage Ratio)')
ax.set_title('Fonctions de vulnérabilité — Tempêtes européennes')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 70)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 5. Impact — Calcul des pertes

On compare les deux fonctions de vulnérabilité (Schwierz vs Welker) sur le même portefeuille et le même aléa.

In [ ]:
results = {}

for name, impf in [('Schwierz', impf_schwierz_custom), ('Welker', impf_welker_custom)]:
    # Mettre l'id à 1 pour correspondre à impf_WS=1 dans l'exposition
    impf_use = ImpactFunc(
        id=1, haz_type='WS',
        intensity=impf.intensity, mdd=impf.mdd, paa=impf.paa,
        intensity_unit='m/s', name=name
    )
    impf_set = ImpactFuncSet([impf_use])
    
    imp = Impact()
    imp.calc(exposure, impf_set, storm_haz, save_mat=True)
    
    results[name] = {
        'eai': imp.aai_agg,
        'max_event': imp.at_event.max(),
        'freq_curve': imp.calc_freq_curve(),
        'impact': imp
    }
    
    print(f"\n--- {name} ---")
    print(f"  EAI (Expected Annual Impact) : {imp.aai_agg:,.0f} USD")
    print(f"  Perte max événement :           {imp.at_event.max():,.0f} USD")
    print(f"  Nb événements avec pertes :     {(imp.at_event > 0).sum()}/{n_storms}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Courbes de fréquence
for name, res in results.items():
    fc = res['freq_curve']
    axes[0].plot(fc.return_per, fc.impact, linewidth=2, label=name)
axes[0].set_xlabel('Période de retour (ans)')
axes[0].set_ylabel('Perte (USD)')
axes[0].set_title('Courbes de fréquence')
axes[0].set_xscale('log')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. EAI comparaison
eai_vals = [res['eai'] for res in results.values()]
bars = axes[1].bar(results.keys(), eai_vals, color=['steelblue', 'coral'], alpha=0.8)
axes[1].set_ylabel('EAI (USD/an)')
axes[1].set_title('Expected Annual Impact')
for bar, val in zip(bars, eai_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02, 
                 f'{val:,.0f}', ha='center', va='bottom', fontsize=9)

# 3. Distribution des pertes par événement
imp_s = results['Schwierz']['impact']
imp_w = results['Welker']['impact']
axes[2].scatter(imp_s.at_event / 1e6, imp_w.at_event / 1e6, alpha=0.6, s=40)
max_val = max(imp_s.at_event.max(), imp_w.at_event.max()) / 1e6 * 1.1
axes[2].plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='1:1')
axes[2].set_xlabel('Perte Schwierz (M USD)')
axes[2].set_ylabel('Perte Welker (M USD)')
axes[2].set_title('Comparaison par événement')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Analyse spatiale des pertes

Distribution géographique de l'impact annuel moyen (EAI) par actif.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for idx, (name, res) in enumerate(results.items()):
    imp = res['impact']
    eai_exp = imp.eai_exp
    
    gdf_plot = exposure.gdf.copy()
    gdf_plot['eai'] = eai_exp
    
    sc = axes[idx].scatter(
        gdf_plot.geometry.x, gdf_plot.geometry.y,
        c=gdf_plot['eai'], s=10, cmap='Reds', vmin=0
    )
    axes[idx].set_title(f'EAI par actif — {name}')
    axes[idx].set_xlabel('Longitude')
    axes[idx].set_ylabel('Latitude')
    plt.colorbar(sc, ax=axes[idx], label='EAI (USD/an)', shrink=0.7)

plt.tight_layout()
plt.show()

## 7. Synthèse

| Métrique | Schwierz | Welker |
|----------|----------|--------|
| Type de courbe | Sigmoïde | Cubique |
| Seuil dommage | ~15 m/s | 25 m/s |
| Comportement vents forts | Saturation rapide | Croissance forte |
| Usage recommandé | Historique suisse | Europe générale |

### Points clés
- Les deux fonctions donnent des résultats significativement différents → le choix de la vulnérabilité est **critique**
- Le SSI permet de comparer la sévérité des tempêtes indépendamment de l'exposition
- Les données WISC (Copernicus) fournissent ~150 footprints historiques calibrés
- Pour une analyse opérationnelle : utiliser `StormEurope.from_footprints()` avec les données WISC

### Prochaines étapes
- Télécharger les footprints WISC réels depuis CDS
- Utiliser LitPop pour une exposition réaliste à haute résolution
- Explorer le clustering temporel des tempêtes (séries de tempêtes)
- Coupler avec les projections CMIP6 pour le changement climatique